# 03. 환경음 이해와 이벤트 기반 상황 추론

환경음 이해는 단순 분류보다 `언제 어떤 소리가 났는지`, `음성 명령과 충돌하는지`, `안전 관련 신호인지`가 중요합니다.
이 노트북에서는 합성 이벤트 시퀀스를 만들고, 이벤트 분류와 시간 구간 요약을 연습합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
events = ["alarm", "door", "keyboard", "speech", "silence"]
event_to_freq = {"alarm": 1200, "door": 160, "keyboard": 3200, "speech": 550, "silence": 0}
sr = 8_000
win = 0.5
t = np.linspace(0, win, int(sr * win), endpoint=False)

def synth_event(label):
    if label == "silence":
        y = np.random.normal(0, 0.01, len(t))
    else:
        freq = event_to_freq[label]
        y = np.sin(2 * np.pi * freq * t) * np.hanning(len(t))
        y += np.random.normal(0, 0.05, len(t))
        if label == "keyboard":
            y *= (np.random.rand(len(t)) > 0.7)
    return y

rows, audio_bank = [], []
for label in events:
    for i in range(60):
        y = synth_event(label)
        audio_bank.append(y)
        spectrum = np.abs(np.fft.rfft(y))
        freqs = np.fft.rfftfreq(len(y), 1 / sr)
        rows.append({
            "label": label,
            "energy": float(np.mean(y ** 2)),
            "peak_freq": float(freqs[np.argmax(spectrum)]),
            "spectral_mean": float(np.mean(spectrum)),
            "spectral_std": float(np.std(spectrum)),
        })

snd = pd.DataFrame(rows)
snd.sample(8, random_state=42)


In [ ]:
try:
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report

    X = snd.drop(columns=["label"])
    y = snd["label"]
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y, cv=cv)
    clf.fit(X, y)
    pred = clf.predict(X)
    print("5-fold accuracy:", scores.round(3), "mean=", scores.mean().round(3))
    print(classification_report(y, pred))
    save_json("environment_sound_cv.json", {"scores": scores.tolist(), "mean": float(scores.mean())})
except Exception as exc:
    print("scikit-learn 설치 후 실행하세요:", exc)


In [ ]:
timeline = [
    (0.0, "speech", "작업 시작"),
    (1.2, "keyboard", "입력 중"),
    (2.0, "alarm", "안전 경고"),
    (2.6, "speech", "계속 진행"),
    (3.1, "door", "출입문 열림"),
]

def risk_policy(timeline):
    risk = []
    for ts, label, note in timeline:
        if label == "alarm":
            risk.append((ts, "STOP", "alarm_detected"))
        elif label == "speech" and risk and risk[-1][1] == "STOP":
            risk.append((ts, "REQUIRE_CONFIRMATION", "speech_after_alarm"))
    return risk

policy_result = pd.DataFrame(risk_policy(timeline), columns=["time", "action", "reason"])
policy_result.to_csv(ARTIFACTS / "sound_policy_events.csv", index=False, encoding="utf-8-sig")
policy_result


## 논문 연결

최종 시스템에서 환경음은 별도 답변을 만드는 모델이 아니라 `신뢰도 게이트`로 사용합니다.
예: 알람이 감지되면 음성 명령의 실행 답변을 바로 수행하지 않고 확인 질문을 생성합니다.
